# TURN-Mapper — evaluate S2 vs S1 vs classical on the 12 synthetic drawings (CPU is enough)

**Inputs:** the output of `turn-segmenter-s2-finetune` (has `out_s2/turn_seg_s2.pt`) and of `turn-segmenter-cubicasa-pretrain` (S1, `out/turn_seg_cubicasa.pt`). Accelerator: None. Internet: On.

Promotion rule (docs/MASTER_PLAN.md): learned becomes the default only if, per case, it is at least as good as classical on 10/12 cases (and later on the FAST A block). Here "at least as good" = room recall, door recall and routes found each >= classical, and wall-crossing edges <= classical.

In [ ]:
!git clone -q https://github.com/ZakiNabeel/Turn-Mapper.git /kaggle/working/Turn-Mapper
%cd /kaggle/working/Turn-Mapper
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!pip -q install -e ".[ml]"
!git log --oneline -1
import glob
S2 = [p for p in glob.glob('/kaggle/input/**/turn_seg_s2.pt', recursive=True)][0]
S1 = [p for p in glob.glob('/kaggle/input/**/turn_seg_cubicasa.pt', recursive=True) if 'misaligned' not in p][0]
print(S2, S1)

In [ ]:
import os, subprocess
for name, seg, w in [('classical', 'classical', None), ('s1', 'learned', S1), ('s2', 'learned', S2)]:
    env = dict(os.environ, **({'TURN_SEG_WEIGHTS': w} if w else {}))
    r = subprocess.run(['python', 'eval/evaluate_synthetic.py', '--segmenter', seg, '--out', f'/kaggle/working/synth_{name}'],
                       env=env, capture_output=True, text=True)
    print(name, 'exit', r.returncode); print(r.stdout[-2500:]); print(r.stderr[-1500:])

In [ ]:
import json
R = {n: {(m['layout'], m['style']): m for m in json.load(open(f'/kaggle/working/synth_{n}/metrics.json'))} for n in ('classical', 's1', 's2')}
def ge(a, c):
    return (a['room_recall'] >= c['room_recall'] and a['door_recall'] >= c['door_recall']
            and a['route_found'] >= c['route_found'] and a['wall_crossing_edges'] <= c['wall_crossing_edges'])
rows, score = [], {'s1': 0, 's2': 0}
print('| case | classical rooms R/P, doors R/P, routes | S1 | S2 | S1>=cls | S2>=cls |')
for k, c in R['classical'].items():
    f = lambda m: f"{m['room_recall']}/{m['room_precision']}, {m['door_recall']}/{m['door_precision']}, {m['route_found']}"
    a, b = R['s1'][k], R['s2'][k]
    g1, g2 = ge(a, c), ge(b, c); score['s1'] += g1; score['s2'] += g2
    rows.append({'case': k, 'classical': c, 's1': a, 's2': b, 's1_ge': g1, 's2_ge': g2})
    print(f"| {k[0]} {k[1]} | {f(c)} | {f(a)} | {f(b)} | {g1} | {g2} |")
print('cases at least as good as classical:', score, 'of', len(R['classical']))
json.dump({'score': score, 'rows': [{**r, 'case': list(r['case'])} for r in rows]}, open('/kaggle/working/compare_synthetic.json', 'w'), indent=1)